# Pondra quick start

A lake on this machine, written to and queried from Python, with a view that keeps itself current
and a point-in-time join. Runs anywhere `pip` does: a laptop, Colab, a cloud notebook.

| Responsibility | Cells | What it does |
|---|---|---|
| R1 — Install | R1-C1 | Installs the `pondra` binary and its Python client |
| R2 — Start a lake | R2-C1 | Starts a node on `./lake`, here, and connects to it |
| R3 — Write | R3-C1, R3-C2 | Creates tables and a view; appends rows exactly once, from Python and from SQL |
| R4 — Query | R4-C1 | SQL into pandas |
| R5 — Stream | R5-C1, R5-C2 | A view that updates as rows arrive; the new rows as they commit |
| R6 — Point in time | R6-C1 | Each order priced as of its own time (`ASOF JOIN`) |
| R7 — Tidy up | R7-C1 | Stops the node; the lake stays in `./lake` |


In [ ]:
# === R1-C1 — Install ===
# The `pondra` wheel carries the binary for this platform and the Python client; pyarrow and
# pandas turn answers into DataFrames. (PONDRA_WHEEL: a wheel file, to try one before PyPI has it.)
import os
package = os.environ.get("PONDRA_WHEEL", "pondra")
%pip install -q $package pyarrow pandas

In [ ]:
# === R2-C1 — Start a lake ===
# pondra.local() starts a node on ./lake (a folder here, or s3://bucket/prefix) and connects to it.
# The node stops when this notebook does; the lake stays, and any other node can open it.
import pondra
db = pondra.local("lake")
db.url

In [ ]:
# === R3-C1 — Create the tables, and a view ===
# An append table of orders, one of price changes to look prices up in, and a view that sums
# the orders per item. A view runs over each batch of new rows as it commits, from its creation
# on. (Running these again changes nothing.)
db.sql("CREATE TABLE IF NOT EXISTS orders (id BIGINT, item VARCHAR, qty BIGINT, ts TIMESTAMP)")
db.sql("CREATE TABLE IF NOT EXISTS prices (item VARCHAR, price DOUBLE, ts TIMESTAMP)")
db.view("items_sold", "SELECT item, sum(qty) AS sold FROM orders GROUP BY item")

In [ ]:
# === R3-C2 — Write rows ===
# append() is exactly-once: a retry after a lost answer is recognised, not applied twice.
# INSERT … SELECT writes straight to Parquet, for bulk loads.
db.append("prices", [{"item": "tea", "price": 3.0, "ts": "2026-01-01T09:00:00"},
                     {"item": "tea", "price": 3.5, "ts": "2026-01-01T12:00:00"},
                     {"item": "cake", "price": 5.0, "ts": "2026-01-01T09:00:00"}])
db.append("orders", [{"id": 1, "item": "tea", "qty": 2, "ts": "2026-01-01T10:00:00"},
                     {"id": 2, "item": "tea", "qty": 1, "ts": "2026-01-01T13:00:00"}])
db.sql("INSERT INTO orders SELECT 2 + value, 'cake', value % 3 + 1, TIMESTAMP '2026-01-01 11:00:00' FROM generate_series(1, 1000)")

In [ ]:
# === R4-C1 — Query into pandas ===
# Every query reads the Parquet files and the rows just written, as one snapshot.
db.sql("SELECT item, count(*) AS orders, sum(qty) AS items FROM orders GROUP BY item ORDER BY item").to_pandas()

In [ ]:
# === R5-C1 — A view that keeps itself current ===
# items_sold was updated in the same commit as each batch of orders; one more order shows it.
before = db.sql("SELECT * FROM items_sold ORDER BY item").to_pandas()
db.append("orders", [{"id": 5000, "item": "tea", "qty": 10, "ts": "2026-01-01T14:00:00"}])
before.merge(db.sql("SELECT * FROM items_sold ORDER BY item").to_pandas(), on="item", suffixes=("_before", "_after"))

In [ ]:
# === R5-C2 — The new rows as they commit ===
# watch() yields each new row the moment it commits (here: replayed from the start of the log).
import itertools
list(itertools.islice(db.watch("orders", after=0), 3))

In [ ]:
# === R6-C1 — Each order priced as of its own time ===
# ASOF JOIN finds, for every order, the latest price of its item at or before the order.
db.sql("""SELECT o.id, o.item, o.ts, p.price, o.qty * p.price AS total
          FROM orders o ASOF JOIN prices p MATCH_CONDITION (o.ts >= p.ts) ON o.item = p.item
          WHERE o.id <= 3 OR o.id = 5000 ORDER BY o.id""").to_pandas()

In [ ]:
# === R7-C1 — Tidy up ===
# Stops the node (it hands the lake on at once). The lake stays in ./lake: `pondra lake` opens a
# SQL shell on it, and `pondra serve --dir lake` serves it again.
db.close()